# 小波同步压缩：瞬时频率、阈值与复系数重分配

本单元直接消费 [c01 小波变换](c01_WaveletTransform.ipynb) 的复系数、导数及核配置，不重新计算 CWT，也不改变上游输入尺度。默认 `method='formal'` 使用尺度求积权重及频率区间宽度；`legacy_sum` 提供不加这两项权重的复系数累加比较。两者共用当前定义的频率估计、阈值和排除规则，不将比较项称为整个旧实现的逐字复现。

|函数|职责|
|---|---|
|`build_sst_frequency_grid`|以周期上下限、区间数及线性／对数频率间隔生成固定频率网格|
|`estimate_instantaneous_frequency`|识别 c01 的导数约定，逐点应用相对或绝对阈值，返回频率及逐尺度诊断|
|`compute_sst`|只在当前采样点沿尺度重分配复系数，返回复谱、逐尺度归属及每点状态|

周期单位为采样点；频率单位为每采样点的周数。计算函数不绘图。完整序列的递推状态调度、来源时刻对齐及独立单图绘制归后续 c03。因果 c01 的历史输出不因追加输入修订，本单元也不读取其他采样点来计算当前点阈值或频率。

In [1]:
from pathlib import Path
import sys
project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")


## 正式定义与离散约定

在 c01 的 L² 尺度归一化下，连续导数对应的周频率估计为 `Im(D/W)/(2pi)`。默认离散同步压缩为
$$T_f[n,\ell]=\frac{1}{\Delta f_\ell}\sum_{k:\widehat f[n,k]\in B_\ell}
W[n,k]\,a_k^{-3/2}\,\Delta a_k.$$
这沿用 Daubechies、Lu、Wu 的一阶小波 SST 定义及尺度到频率的重分配，参见 §2、式 (2.2)、(2.3)。本实现改以周频率 `f=omega/(2pi)` 表示密度，因此使用 `Delta f`。[论文全文](https://arxiv.org/pdf/0912.2437)

`Delta a` 采用 c01 在既定尺度列表上的梯形求积权重；不按当前可用尺度重新估算，否则预热期间会改变通道的含义。线性和对数频率网格都使用各自真实区间宽度 `Delta f`，对数网格不把宽度当常数。

因果 Gammatone 的 `D` 为 `bilinear_model_derivative`：其离散复指数满足 `D/W=i·2tan(pi f)`，故使用 `atan(Im(D/W)/2)/pi` 逆畸变。Morlet 的 `sampled_continuous_kernel_derivative` 使用 `Im(D/W)/(2pi)`。这是针对 c01 离散化的频率估计选择；不声称一般调幅调频信号的离散估计与连续导数严格相等。

频率区间内部采用 `[left, right)`，最后一个区间包含最右端。负频率、零频率、Nyquist 及以上频率和网格以外的贡献均明确排除，不把频率裁剪到边缘。网格参数只由调用者配置，不根据整段数据的极值改变。

`legacy_sum` 使用同样的区间归属，累加 `W`，不乘尺度权重、不除区间宽度。它的数值单位与正式密度不同，不能直接拿幅值大小判断哪种方法更好。

In [2]:
from dataclasses import dataclass
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow as pa
from config.data_contracts import pandas_to_arrow
from R04_Research.a01_Methods.b06_WaveletAnalysis.c01_WaveletTransform import (
    WaveletBank, build_wavelet_bank, WAVELET_CHANNEL_SCHEMA, WAVELET_COEFFICIENT_SCHEMA,
)

SYNCHROSQUEEZING_VERSION = "0.1.0"

@dataclass(frozen=True)
class SSTFrequencyGrid:
    edges: np.ndarray
    bins_df: pd.DataFrame
    parameters: dict
    signature: str

def build_sst_frequency_grid(*, period_min_samples, period_max_samples, n_bins=128,
                             spacing='linear_frequency') -> SSTFrequencyGrid:
    """固定正频率区间，周期边界包含，n_bins 为区间数。网格不依赖输入数据。"""
    if np.iscomplexobj([period_min_samples,period_max_samples]):
        raise ValueError('周期边界必须为实数')
    lower_period,upper_period = float(period_min_samples),float(period_max_samples)
    if not np.isfinite([lower_period,upper_period]).all() or not 2<lower_period<upper_period:
        raise ValueError('必须满足 2 < period_min_samples < period_max_samples')
    if isinstance(n_bins,(bool,np.bool_)) or not isinstance(n_bins,(int,np.integer)) or n_bins<1:
        raise ValueError('n_bins 必须为正整数')
    if spacing=='linear_frequency':
        edges = np.linspace(1/upper_period,1/lower_period,n_bins+1)
    elif spacing=='log_frequency':
        edges = np.geomspace(1/upper_period,1/lower_period,n_bins+1)
    else:
        raise ValueError('spacing 必须为 linear_frequency 或 log_frequency')
    widths = np.diff(edges)
    if not np.isfinite(edges).all() or (widths<=0).any():
        raise ValueError('当前数值精度下不能生成严格递增的频率网格')
    centers = (edges[:-1]+edges[1:])/2
    bins_df = pd.DataFrame(dict(bin_id=np.arange(n_bins,dtype=np.int64),
        frequency_left=edges[:-1],frequency_right=edges[1:],frequency_center=centers,
        frequency_width=widths,period_samples=1/centers))
    parameters = dict(period_min_samples=lower_period,period_max_samples=upper_period,
        n_bins=int(n_bins),spacing=spacing,frequency_unit='cycles_per_sample',
        bin_boundary='left_closed_right_open_last_right_closed')
    signature = hashlib.sha256(json.dumps(dict(version=SYNCHROSQUEEZING_VERSION,**parameters),sort_keys=True).encode()).hexdigest()
    edges.setflags(write=False)
    return SSTFrequencyGrid(edges,bins_df,parameters,signature)

def estimate_instantaneous_frequency(coefficients, derivatives, bank: WaveletBank, *,
        transform_status=None, transform_reason=None, threshold_mode='relative',
        threshold_ratio=0.01, absolute_threshold=None) -> dict:
    """返回 time×channel 数组；一维输入表示一个点。阈值只使用当前行。

    相对阈值=ratio*max(|W|)，最大值只取当前点已可用且有限的 CWT 系数。
    |W|>=threshold 才参与估计；W=0 的频率未定义，即使阈值为零也排除。
    raw_frequency 保留有限的物理范围外估计；frequencies 只保留 (0,0.5) 内的值。
    """
    W,D = np.asarray(coefficients,dtype=complex),np.asarray(derivatives,dtype=complex)
    if W.ndim==1:
        W = W[None,:]
    if D.ndim==1:
        D = D[None,:]
    if W.ndim!=2 or D.shape!=W.shape or W.shape[1]!=len(bank.channels_df):
        raise ValueError('复系数、导数必须同形，列数必须与核组通道数一致')
    if not bank.parameters.get('supports_sst',False):
        raise ValueError('此核组不支持复系数 SST')
    convention = bank.parameters.get('derivative_convention')
    if convention not in ('bilinear_model_derivative','sampled_continuous_kernel_derivative'):
        raise ValueError('未实现的导数约定，不能猜测频率映射')
    if threshold_mode not in ('relative','absolute'):
        raise ValueError('threshold_mode 必须为 relative 或 absolute')
    if not np.isscalar(threshold_ratio) or np.iscomplexobj(threshold_ratio) or not np.isfinite(threshold_ratio) or threshold_ratio<0:
        raise ValueError('threshold_ratio 必须为有限非负实数')
    if threshold_mode=='absolute':
        if absolute_threshold is None or not np.isscalar(absolute_threshold) or np.iscomplexobj(absolute_threshold) or not np.isfinite(absolute_threshold) or absolute_threshold<0:
            raise ValueError('绝对模式必须传入有限非负 absolute_threshold')
    elif absolute_threshold is not None:
        raise ValueError('传入 absolute_threshold 时请显式设置 threshold_mode=absolute')
    status = np.full(W.shape,'ok',dtype=object) if transform_status is None else np.asarray(transform_status,dtype=object).copy()
    reason = np.full(W.shape,'',dtype=object) if transform_reason is None else np.asarray(transform_reason,dtype=object).copy()
    if status.ndim==1:
        status = status[None,:]
    if reason.ndim==1:
        reason = reason[None,:]
    if status.shape!=W.shape or reason.shape!=W.shape:
        raise ValueError('上游状态、原因必须与复系数同形')
    if not np.isin(status,['ok','warmup','boundary_incomplete','failed']).all():
        raise ValueError('不支持的上游变换状态')
    upstream_available = status=='ok'
    with np.errstate(over='ignore',invalid='ignore'):
        amplitude = abs(W)
    finite_coefficient = np.isfinite(W)&np.isfinite(amplitude)
    invalid_coefficient = upstream_available&~finite_coefficient
    status[invalid_coefficient],reason[invalid_coefficient] = 'failed','nonfinite_coefficient'
    available = upstream_available&finite_coefficient
    maximum = np.max(np.where(available,amplitude,0.0),axis=1)
    maximum[~available.any(axis=1)] = np.nan
    with np.errstate(over='ignore',invalid='ignore'):
        threshold = maximum*float(threshold_ratio) if threshold_mode=='relative' else np.full(len(W),float(absolute_threshold))
    threshold[~available.any(axis=1)] = np.nan
    invalid_threshold = available&~np.isfinite(threshold[:,None])
    status[invalid_threshold],reason[invalid_threshold] = 'failed','nonfinite_threshold'
    zero = (status=='ok')&(amplitude==0)
    status[zero],reason[zero] = 'excluded','zero_coefficient'
    below = (status=='ok')&(amplitude<threshold[:,None])
    status[below],reason[below] = 'excluded','below_threshold'
    invalid_derivative = (status=='ok')&~np.isfinite(D)
    status[invalid_derivative],reason[invalid_derivative] = 'failed','nonfinite_derivative'
    candidates = status=='ok'
    ratio = np.full(W.shape,complex(np.nan,np.nan))
    with np.errstate(over='ignore',divide='ignore',invalid='ignore'):
        np.divide(D,W,out=ratio,where=candidates)
    invalid_ratio = candidates&~np.isfinite(ratio)
    status[invalid_ratio],reason[invalid_ratio] = 'failed','nonfinite_derivative_ratio'
    candidates = status=='ok'
    model_angular_frequency = np.where(candidates,ratio.imag,np.nan)
    raw_frequency = (np.arctan(model_angular_frequency/2)/np.pi if convention=='bilinear_model_derivative'
        else model_angular_frequency/(2*np.pi))
    nonpositive = candidates&(raw_frequency<=0)
    status[nonpositive],reason[nonpositive] = 'excluded','nonpositive_frequency'
    high = candidates&(raw_frequency>=0.5)
    status[high],reason[high] = 'excluded','at_or_above_nyquist'
    invalid_frequency = candidates&~np.isfinite(raw_frequency)
    status[invalid_frequency],reason[invalid_frequency] = 'failed','nonfinite_frequency'
    frequencies = np.where(status=='ok',raw_frequency,np.nan)
    periods = np.full(W.shape,np.nan)
    np.divide(1.,frequencies,out=periods,where=status=='ok')
    return dict(frequencies=frequencies,raw_frequency=raw_frequency,period_samples=periods,
        model_angular_frequency=model_angular_frequency,amplitude=amplitude,
        threshold=threshold,max_available_amplitude=maximum,status=status,reason=reason,
        bank_signature=bank.signature,derivative_convention=convention,
        parameters=dict(threshold_mode=threshold_mode,threshold_ratio=float(threshold_ratio),
            absolute_threshold=None if absolute_threshold is None else float(absolute_threshold)))


## 阈值、状态与可检验关系

默认 `threshold_mode='relative', threshold_ratio=0.01`，阈值为当前点可用尺度中最大 `|W|` 的比例，比例可为任意有限非负实数。绝对模式须显式传入 `absolute_threshold`，单位与 CWT 系数一致。不使用整段序列或未来点的最大值。

`estimate_instantaneous_frequency` 接受同形的 `coefficients`／`derivatives`、对应 `bank`，以及可选的上游 `transform_status`／`transform_reason`；二维形状为采样点 × 尺度，一维表示单点。省略上游状态时，所有系数按待检查的可用输入处理；使用 c01 产物应传入其状态与原因以保留预热和失败。返回的 `frequencies`、`raw_frequency`、`period_samples`、`status`、`reason` 均为二维，`threshold` 和 `max_available_amplitude` 每点一个值。`raw_frequency` 保留有限但物理范围无效的估计；有效频率字段对排除项为空。

`compute_sst` 接受 c01 返回的 `wavelet_transform` 字典（至少包含 `bank`、`coefficients`、`derivatives`）及 `SSTFrequencyGrid`，并将其中 `status`／`reason` 传递到频率估计。返回 `spectrum`（采样点 × 频率区间复数矩阵）、`instantaneous_frequency`、`assigned_bins`（采样点 × 尺度，未纳入为 -1）、`reassignment_status`／`reassignment_reason` 和 `diagnostics_df`。正式模式要求至少两个尺度且固定求积权重为有限正数；比较模式可以接受单尺度。计算不修改输入数组。

每个通道保留上游预热、边界不足及失败。零系数、低于阈值、物理频率无效及网格以外是不同排除原因。部分尺度尚未就绪时，输出 `partial`，表示只包含已可用尺度的贡献；全部尺度不可用时谱为空。全部尺度已经可用而没有任何合格贡献时，谱为有效的零，不能把零谱与缺失谱混淆。

正式定义满足本实现的有限求积恒等式：
$$\sum_\ell T_f[n,\ell]\Delta f_\ell=\sum_{k\in\text{实际纳入}}W[n,k]a_k^{-3/2}\Delta a_k.$$
这是复系数积分恒等式，不是 CWT 与 SST 的能量相等，也不证明原信号已经重构。`legacy_sum` 的对应关系为直接对频率区间求和。近似解析核、有限尺度范围、阈值以及被排除的频率共同限定结果；本单元没有实现原信号或分量重构。

In [3]:
def compute_sst(wavelet_transform: dict, frequency_grid: SSTFrequencyGrid, *, method='formal',
        threshold_mode='relative', threshold_ratio=0.01, absolute_threshold=None) -> dict:
    """只沿每点的尺度重分配，返回复谱 time×bin、归属及点诊断；不绘图。

    无可用尺度的行置空；缺部分尺度/局部数值失败的行标 partial；全部就绪而无
    纳入贡献的行保留零谱。正式权重在核组建立时固定，不随当前就绪尺度改变。
    """
    if method not in ('formal','legacy_sum'):
        raise ValueError('method 必须为 formal 或 legacy_sum')
    bank = wavelet_transform['bank']
    W = np.asarray(wavelet_transform['coefficients'],dtype=complex)
    if W.ndim==1:
        W = W[None,:]
    if method=='formal':
        weights = bank.channels_df['scale_weight'].to_numpy(dtype=float)
        if len(weights)<2 or not np.isfinite(weights).all() or (weights<=0).any():
            raise ValueError('正式 SST 至少需要两个尺度，且固定尺度求积权重必须有限、为正')
    else:
        weights = np.ones(len(bank.channels_df))
    instantaneous = estimate_instantaneous_frequency(W,wavelet_transform['derivatives'],bank,
        transform_status=wavelet_transform.get('status'),transform_reason=wavelet_transform.get('reason'),
        threshold_mode=threshold_mode,threshold_ratio=threshold_ratio,absolute_threshold=absolute_threshold)
    upstream_status = np.asarray(wavelet_transform.get('status',np.full(W.shape,'ok')),dtype=object)
    if upstream_status.ndim==1:
        upstream_status = upstream_status[None,:]
    frequencies = instantaneous['frequencies']
    edges = frequency_grid.edges
    widths = np.diff(edges)
    bin_count = len(widths)
    if bin_count<1 or not np.isfinite(edges).all() or (widths<=0).any() or edges[0]<=0 or edges[-1]>=0.5:
        raise ValueError('频率网格必须为 (0,0.5) 内的严格递增边界')
    inside = (instantaneous['status']=='ok')&(frequencies>=edges[0])&(frequencies<=edges[-1])
    assigned_bins = np.full(W.shape,-1,dtype=np.int64)
    assigned_bins[inside] = np.searchsorted(edges,frequencies[inside],side='right')-1
    # 最后一条右边界本身被包含；未将任何范围外频率裁剪入边缘区间。
    assigned_bins[inside&(assigned_bins==bin_count)] = bin_count-1
    reassign_status,reassign_reason = instantaneous['status'].copy(),instantaneous['reason'].copy()
    outside = (instantaneous['status']=='ok')&~inside
    reassign_status[outside],reassign_reason[outside] = 'excluded','outside_frequency_grid'
    rows,channels = np.nonzero(inside)
    spectrum = np.zeros((len(W),bin_count),dtype=complex)
    with np.errstate(over='ignore',invalid='ignore',divide='ignore'):
        contributions = W[rows,channels]*weights[channels]
        np.add.at(spectrum,(rows,assigned_bins[rows,channels]),contributions)
        if method=='formal':
            spectrum /= widths[None,:]
    computable = np.isin(instantaneous['status'],['ok','excluded']).any(axis=1)
    ready_count = (upstream_status=='ok').sum(axis=1)
    unavailable_count = (upstream_status!='ok').sum(axis=1)
    failed_count = (instantaneous['status']=='failed').sum(axis=1)
    point_status = np.full(len(W),'ok',dtype=object)
    point_reason = np.full(len(W),'',dtype=object)
    partial = computable&((unavailable_count>0)|(failed_count>0))
    point_status[partial],point_reason[partial] = 'partial','unavailable_or_failed_scales'
    no_usable = ~computable
    all_warmup = no_usable&(upstream_status=='warmup').all(axis=1)
    all_boundary = no_usable&(upstream_status=='boundary_incomplete').all(axis=1)
    point_status[no_usable],point_reason[no_usable] = 'failed','no_computable_scales'
    point_status[all_warmup],point_reason[all_warmup] = 'warmup','insufficient_history'
    point_status[all_boundary],point_reason[all_boundary] = 'boundary_incomplete','incomplete_kernel_support'
    all_upstream_failed = no_usable&(upstream_status=='failed').all(axis=1)
    point_reason[all_upstream_failed] = 'upstream_failed'
    integration_width = widths if method=='formal' else np.ones(bin_count)
    with np.errstate(over='ignore',invalid='ignore'):
        retained = (spectrum*integration_width[None,:]).sum(axis=1)
    numerical_failure = computable&(~np.isfinite(spectrum).all(axis=1)|~np.isfinite(retained))
    point_status[numerical_failure],point_reason[numerical_failure] = 'failed','nonfinite_sst_accumulation'
    spectrum[no_usable|numerical_failure] = complex(np.nan,np.nan)
    retained[no_usable|numerical_failure] = complex(np.nan,np.nan)
    diagnostics_df = pd.DataFrame(dict(
        max_available_amplitude=instantaneous['max_available_amplitude'],coefficient_threshold=instantaneous['threshold'],
        ready_scale_count=ready_count,unavailable_scale_count=unavailable_count,failed_scale_count=failed_count,
        reassigned_scale_count=inside.sum(axis=1),excluded_scale_count=(reassign_status=='excluded').sum(axis=1),
        outside_grid_scale_count=outside.sum(axis=1),
        below_threshold_scale_count=(reassign_reason=='below_threshold').sum(axis=1),
        zero_scale_count=(reassign_reason=='zero_coefficient').sum(axis=1),
        nonpositive_scale_count=(reassign_reason=='nonpositive_frequency').sum(axis=1),
        nyquist_scale_count=(reassign_reason=='at_or_above_nyquist').sum(axis=1),
        retained_coefficient_real=retained.real,retained_coefficient_imag=retained.imag,
        sst_status=point_status,sst_reason=point_reason))
    return dict(bank=bank,frequency_grid=frequency_grid,spectrum=spectrum,
        instantaneous_frequency=instantaneous,assigned_bins=assigned_bins,
        reassignment_status=reassign_status,reassignment_reason=reassign_reason,
        diagnostics_df=diagnostics_df,parameters=dict(method=method,**instantaneous['parameters']),
        is_causal=bool(bank.parameters['is_causal']))


## 保存契约与展示

demo 分别保存频率网格、逐尺度频率／归属和每点复谱。复谱采用宽表：每个采样点一行，`sst_real_000`／`sst_imag_000` 等字段对应频率表的 `bin_id`，避免为零谱重复保存大量坐标行。复谱局部 Arrow Schema 明确区间数、方法和网格标识，生成身份同时明确核及参数；每点状态始终与复谱一起保存。幅值、能量与相位可由复谱计算，零幅值相位未定义。

`sample_at`、`bar_at`、交易日和训练／测试侧原样继承 c01；Morlet 的离线结果仍不宣称当前时点可用。计算不因自然时间跳跃重置；本单元没有滤波状态，也不调用 c01 重新拟合或补值。

In [4]:
SST_GRID_SCHEMA = pa.schema([
    pa.field('bin_id',pa.int64()),pa.field('frequency_left',pa.float64()),pa.field('frequency_right',pa.float64()),
    pa.field('frequency_center',pa.float64()),pa.field('frequency_width',pa.float64()),pa.field('period_samples',pa.float64()),
],metadata={b'method':b'synchrosqueezing',b'schema_version':b'0.1.0',b'frequency_unit':b'cycles_per_sample'})

SST_FREQUENCY_SCHEMA = pa.schema([
    pa.field('sample_id',pa.int64()),pa.field('channel_id',pa.int64()),pa.field('raw_frequency',pa.float64()),
    pa.field('frequency_cycles_per_sample',pa.float64()),pa.field('period_samples',pa.float64()),
    pa.field('model_angular_frequency',pa.float64()),pa.field('coefficient_threshold',pa.float64()),
    pa.field('frequency_status',pa.string()),pa.field('frequency_reason',pa.string()),
    pa.field('assigned_bin_id',pa.int64()),pa.field('reassignment_status',pa.string()),pa.field('reassignment_reason',pa.string()),
],metadata={b'method':b'synchrosqueezing',b'schema_version':b'0.1.0',b'frequency_unit':b'cycles_per_sample'})

SST_POINT_FIELDS = [
    pa.field('sample_id',pa.int64()),pa.field('sample_at',pa.timestamp('us',tz='Asia/Shanghai')),
    pa.field('bar_at',pa.timestamp('us',tz='Asia/Shanghai')),pa.field('trading_date',pa.date32()),pa.field('sample_side',pa.string()),
    pa.field('max_available_amplitude',pa.float64()),pa.field('coefficient_threshold',pa.float64()),
    *[pa.field(name,pa.int64()) for name in ['ready_scale_count','unavailable_scale_count','failed_scale_count',
        'reassigned_scale_count','excluded_scale_count','outside_grid_scale_count','below_threshold_scale_count',
        'zero_scale_count','nonpositive_scale_count','nyquist_scale_count']],
    pa.field('retained_coefficient_real',pa.float64()),pa.field('retained_coefficient_imag',pa.float64()),
    pa.field('sst_status',pa.string()),pa.field('sst_reason',pa.string()),
]


In [5]:
SST_COLUMN_LABELS = dict(
    sample_id='sample_id（采样点编号）',sample_at='sample_at（采样位置时刻）',bar_at='bar_at（源报价可得时刻）',
    trading_date='trading_date（交易日）',sample_side='sample_side（训练测试侧）',channel_id='channel_id（尺度通道编号）',
    bin_id='bin_id（频率区间编号）',frequency_left='frequency_left（区间左频率）',frequency_right='frequency_right（区间右频率）',
    frequency_center='frequency_center（区间中心周频率）',frequency_width='frequency_width（区间频率宽度）',
    period_samples='period_samples（周期采样点数）',raw_frequency='raw_frequency（未裁剪的周频率估计）',
    frequency_cycles_per_sample='frequency_cycles_per_sample（有效瞬时周频率）',
    model_angular_frequency='model_angular_frequency（导数模型角频率）',
    max_available_amplitude='max_available_amplitude（当前点最大可用幅值）',coefficient_threshold='coefficient_threshold（当前点幅值阈值）',
    frequency_status='frequency_status（瞬时频率状态）',frequency_reason='frequency_reason（瞬时频率原因）',
    assigned_bin_id='assigned_bin_id（重分配区间编号）',reassignment_status='reassignment_status（重分配状态）',
    reassignment_reason='reassignment_reason（重分配原因）',sst_status='sst_status（同步压缩状态）',sst_reason='sst_reason（同步压缩原因）',
    ready_scale_count='ready_scale_count（上游就绪尺度数）',unavailable_scale_count='unavailable_scale_count（上游未就绪尺度数）',
    failed_scale_count='failed_scale_count（失败尺度数）',reassigned_scale_count='reassigned_scale_count（纳入重分配尺度数）',
    excluded_scale_count='excluded_scale_count（排除尺度数）',outside_grid_scale_count='outside_grid_scale_count（网格外尺度数）',
    below_threshold_scale_count='below_threshold_scale_count（低于阈值尺度数）',zero_scale_count='zero_scale_count（零系数尺度数）',
    nonpositive_scale_count='nonpositive_scale_count（非正频率尺度数）',nyquist_scale_count='nyquist_scale_count（Nyquist及以上尺度数）',
    retained_coefficient_real='retained_coefficient_real（纳入复系数积分实部）',retained_coefficient_imag='retained_coefficient_imag（纳入复系数积分虚部）',
    kernel_type='kernel_type（核类型）',interval='interval（展示区间）',
)


## 真实 demo：消费完整 c01 RB 小波产物

固定读取 c01 的两种核、周期 8—256、N=60、time、all_contracts、log_interpolate、fixed、上游 log_return 配置。完整范围包含训练 2010-01-01—2024-12-31、测试 2025-01-01—2026-09-30。下列函数直接消费已保存的复系数及导数，不重复执行 CWT。

正式 demo 使用周期边界 8—256、128 个线性频率区间、当前点 0.01 比例阈值；另在内存用同一真实输入核对绝对阈值、对数频率区间及 legacy_sum 的定义。完整数据检查每点纳入／排除归属、复积分恒等式、预热和缺失继承，以及追加数据不改变此前结果。合成数学与边界测试另行记录。

In [6]:
import os
import platform
import scipy
import pyarrow.parquet as pq
from datetime import datetime,timezone
from IPython.display import display
from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256

demo_producer_notebook = candidate_root/'R04_Research/a01_Methods/b06_WaveletAnalysis/c02_Synchrosqueezing.ipynb'
demo_upstream_notebook = demo_producer_notebook.with_name('c01_WaveletTransform.ipynb')
demo_range_token = 'train20100101-20241231_test20250101-20260930'
demo_input_token = 'N60_time_all_contracts_log_interpolate_fixed_log_return'
demo_period_token = 'P8-16-32-64-128-256'
demo_kernel_tokens = dict(causal_gammatone='causal_gammatone_q4_alpha1_omega6_mem1e-6',
    morlet='morlet_sigma1_omega6_support8_none')
demo_source_book = json.loads(demo_upstream_notebook.read_text(encoding='utf-8'))
demo_source_text = '\n\n'.join(''.join(c['source']) for c in demo_source_book['cells']
    if c['cell_type']=='code' and 'demo-dependency' in c.get('metadata',{}).get('tags',[]))
demo_upstream_source_hash = hashlib.sha256(demo_source_text.encode()).hexdigest()
demo_transforms,demo_coordinates,demo_upstream_identities = {},{},{}
for demo_kind,demo_kernel_token in demo_kernel_tokens.items():
    demo_tables,demo_identities = {},{}
    for demo_role,demo_schema in [('channels',WAVELET_CHANNEL_SCHEMA),('coefficients',WAVELET_COEFFICIENT_SCHEMA)]:
        demo_path = demo_upstream_notebook.parent/'c01_WaveletTransform_DemoData'/(
            f'c01_WaveletTransform_RB_{demo_range_token}_{demo_input_token}_{demo_kernel_token}_{demo_period_token}_{demo_role}_demo.parquet')
        demo_table = pq.read_table(demo_path)
        demo_identity = json.loads(demo_table.schema.metadata[b'demo_identity'])
        assert demo_identity['producer_notebook']==demo_upstream_notebook.relative_to(candidate_root).as_posix()
        assert demo_identity['method_source_sha256']==demo_upstream_source_hash
        assert demo_identity['output_role']==demo_role
        assert demo_identity['parameters']['kernel_type']==demo_kind
        assert demo_identity['parameters']['periods']==[8,16,32,64,128,256]
        assert demo_identity['parameters']['input_col']=='log_return'
        assert demo_identity['parameters']['input_transform']=='identity'
        assert demo_identity['parameters']['sampling']==demo_input_token
        assert demo_identity['parameters']['upstream_input_transform']=='global_log_difference'
        assert demo_identity['scope']['mapping_read']==dict(start='2010-01-01',end='2026-09-30')
        assert demo_identity['scope']['test_trading_dates']==dict(start='2025-01-01',end='2026-09-30')
        assert demo_identity['scope']['training_trading_dates']['end']=='2024-12-31'
        assert demo_identity['scope']['timezone']=='Asia/Shanghai'
        for demo_dependency,demo_expected_hash in demo_identity['dependency_method_source_sha256'].items():
            demo_dependency_path = candidate_root/f"R04_Research/a01_Methods/b03_Sampling/{'c01_SamplingPositions' if demo_dependency=='sampling_positions' else 'c02_SamplingExecution'}.ipynb"
            demo_dependency_book = json.loads(demo_dependency_path.read_text(encoding='utf-8'))
            demo_dependency_source = '\n\n'.join(''.join(c['source']) for c in demo_dependency_book['cells']
                if c['cell_type']=='code' and {'export','demo-dependency'}.issubset(c.get('metadata',{}).get('tags',[])))
            assert hashlib.sha256(demo_dependency_source.encode()).hexdigest()==demo_expected_hash
        for demo_role_name,demo_relative in [('notebook_loader','R04_Research/a00_notebook_loader.py'),('data_contracts','config/data_contracts.py')]:
            assert hashlib.sha256((candidate_root/demo_relative).read_bytes()).hexdigest()==demo_identity['dependency_source_sha256'][demo_role_name]
        assert arrow_content_sha256(demo_table,demo_schema,demo_identity['output_sort_keys'])==demo_identity['output_content_sha256']
        demo_tables[demo_role],demo_identities[demo_role] = demo_table,demo_identity
    assert demo_identities['channels']['bank_signature']==demo_identities['coefficients']['bank_signature']
    demo_params = demo_identities['coefficients']['parameters']
    demo_bank = build_wavelet_bank(demo_params['periods'],kernel_type=demo_kind,kernel_params=demo_params['kernel_params'],
        warmup_samples=demo_params['warmup_samples'] if demo_kind=='causal_gammatone' else None,
        memory_tolerance=demo_params['memory_tolerance'])
    assert demo_bank.signature==demo_identities['coefficients']['bank_signature']
    demo_input_df = demo_tables['coefficients'].to_pandas()
    demo_channel_count = len(demo_bank.channels_df)
    demo_W = (demo_input_df['coefficient_real'].to_numpy()+1j*demo_input_df['coefficient_imag'].to_numpy()).reshape(-1,demo_channel_count)
    demo_D = (demo_input_df['derivative_real'].to_numpy()+1j*demo_input_df['derivative_imag'].to_numpy()).reshape(-1,demo_channel_count)
    demo_transforms[demo_kind] = dict(bank=demo_bank,coefficients=demo_W,derivatives=demo_D,
        status=demo_input_df['transform_status'].to_numpy().reshape(demo_W.shape),
        reason=demo_input_df['transform_reason'].to_numpy().reshape(demo_W.shape))
    demo_coordinates[demo_kind] = demo_input_df.iloc[::demo_channel_count][['sample_id','sample_at','bar_at','trading_date','sample_side']].reset_index(drop=True)
    demo_upstream_identities[demo_kind] = demo_identities
    print(f'已核对真实 {demo_kind} 上游：{len(demo_W):,} 点 × {demo_channel_count} 尺度',flush=True)
demo_grid = build_sst_frequency_grid(period_min_samples=8,period_max_samples=256,n_bins=128,spacing='linear_frequency')
demo_results = {}


已核对真实 causal_gammatone 上游：21,316 点 × 6 尺度
已核对真实 morlet 上游：21,316 点 × 6 尺度


In [7]:
for demo_kind,demo_transform in demo_transforms.items():
    demo_independent_frequency = estimate_instantaneous_frequency(demo_transform['coefficients'],demo_transform['derivatives'],demo_transform['bank'],
        transform_status=demo_transform['status'],transform_reason=demo_transform['reason'],threshold_ratio=0.01)
    demo_sst = compute_sst(demo_transform,demo_grid,threshold_ratio=0.01)
    for demo_field in ['frequencies','threshold','status','reason']:
        np.testing.assert_array_equal(demo_independent_frequency[demo_field],demo_sst['instantaneous_frequency'][demo_field])
    demo_included = demo_sst['assigned_bins']>=0
    demo_weights = demo_transform['bank'].channels_df['scale_weight'].to_numpy(dtype=float)
    demo_expected_integral = np.where(demo_included,demo_transform['coefficients']*demo_weights[None,:],0).sum(axis=1)
    demo_good = demo_sst['diagnostics_df']['sst_status'].isin(['ok','partial']).to_numpy()
    np.testing.assert_allclose((demo_sst['spectrum']*np.diff(demo_grid.edges)[None,:]).sum(axis=1)[demo_good],
        demo_expected_integral[demo_good],rtol=1e-12,atol=1e-14)
    assert np.isfinite(demo_sst['spectrum'][demo_good]).all()
    assert np.isnan(demo_sst['spectrum'][~demo_good]).all()
    demo_frequency = demo_sst['instantaneous_frequency']
    demo_amplitude = abs(demo_transform['coefficients'])
    demo_available = (demo_transform['status']=='ok')&np.isfinite(demo_amplitude)
    demo_maximum = np.max(np.where(demo_available,demo_amplitude,0.),axis=1)
    demo_maximum[~demo_available.any(axis=1)] = np.nan
    np.testing.assert_allclose(demo_frequency['threshold'],0.01*demo_maximum,equal_nan=True)
    assert (demo_sst['assigned_bins'][demo_sst['reassignment_status']!='ok']==-1).all()
    assert (demo_sst['assigned_bins'][demo_included]<128).all()
    assert (demo_sst['diagnostics_df']['reassigned_scale_count'].to_numpy()==demo_included.sum(axis=1)).all()
    assert not demo_sst['diagnostics_df']['sst_reason'].eq('nonfinite_sst_accumulation').any()
    demo_results[demo_kind] = demo_sst
    print(f"完整 {demo_kind} 正式 SST：{demo_sst['diagnostics_df']['sst_status'].value_counts().to_dict()}",flush=True)

demo_causal = demo_transforms['causal_gammatone']
demo_prefix_count = int(np.flatnonzero(demo_coordinates['causal_gammatone']['sample_side'].eq('test').to_numpy())[0])
demo_prefix_input = {key:(value[:demo_prefix_count] if isinstance(value,np.ndarray) else value) for key,value in demo_causal.items()}
demo_prefix_sst = compute_sst(demo_prefix_input,demo_grid)
for demo_field in ['spectrum','assigned_bins','reassignment_status','reassignment_reason']:
    np.testing.assert_array_equal(demo_prefix_sst[demo_field],demo_results['causal_gammatone'][demo_field][:demo_prefix_count])
pd.testing.assert_frame_equal(demo_prefix_sst['diagnostics_df'],demo_results['causal_gammatone']['diagnostics_df'].iloc[:demo_prefix_count].reset_index(drop=True))

demo_absolute = compute_sst(demo_causal,demo_grid,threshold_mode='absolute',absolute_threshold=1e-4)
demo_ready = demo_absolute['diagnostics_df']['ready_scale_count']>0
np.testing.assert_array_equal(demo_absolute['instantaneous_frequency']['threshold'][demo_ready],np.full(demo_ready.sum(),1e-4))
demo_log_grid = build_sst_frequency_grid(period_min_samples=8,period_max_samples=256,n_bins=128,spacing='log_frequency')
demo_log_sst = compute_sst(demo_causal,demo_log_grid)
demo_log_mask = demo_log_sst['assigned_bins']>=0
demo_log_good = demo_log_sst['diagnostics_df']['sst_status'].isin(['ok','partial']).to_numpy()
demo_log_expected = np.where(demo_log_mask,demo_causal['coefficients']*demo_causal['bank'].channels_df['scale_weight'].to_numpy()[None,:],0).sum(axis=1)
np.testing.assert_allclose((demo_log_sst['spectrum']*np.diff(demo_log_grid.edges)).sum(axis=1)[demo_log_good],demo_log_expected[demo_log_good],atol=1e-14)
demo_legacy = compute_sst(demo_causal,demo_grid,method='legacy_sum')
demo_legacy_mask = demo_legacy['assigned_bins']>=0
demo_legacy_good = demo_legacy['diagnostics_df']['sst_status'].isin(['ok','partial']).to_numpy()
np.testing.assert_allclose(demo_legacy['spectrum'].sum(axis=1)[demo_legacy_good],
    np.where(demo_legacy_mask,demo_causal['coefficients'],0).sum(axis=1)[demo_legacy_good],atol=1e-14)
print('完整真实输入的复积分、逐点阈值、范围排除、历史不变性及并列计算口径验收通过',flush=True)


完整 causal_gammatone 正式 SST：{'ok': 16055, 'partial': 5074, 'warmup': 186, 'failed': 1}
完整 morlet 正式 SST：{'ok': 17403, 'partial': 3788, 'boundary_incomplete': 124, 'failed': 1}
完整真实输入的复积分、逐点阈值、范围排除、历史不变性及并列计算口径验收通过


In [8]:
demo_book = json.loads(demo_producer_notebook.read_text(encoding='utf-8'))
demo_method_source = '\n\n'.join(''.join(c['source']) for c in demo_book['cells']
    if c['cell_type']=='code' and 'demo-dependency' in c.get('metadata',{}).get('tags',[]))
demo_output_dir = demo_producer_notebook.parent/'c02_Synchrosqueezing_DemoData'
demo_output_dir.mkdir(parents=True,exist_ok=True)
demo_point_schema = pa.schema([*SST_POINT_FIELDS,*[
    pa.field(f'sst_{component}_{j:03d}',pa.float64()) for component in ['real','imag'] for j in range(128)]],
    metadata={b'method':b'synchrosqueezing',b'schema_version':b'0.1.0',b'n_bins':b'128',b'frequency_grid_signature':demo_grid.signature.encode()})
demo_frequency_frames,demo_point_frames = {},{}
for demo_kind,demo_sst in demo_results.items():
    demo_coords = demo_coordinates[demo_kind]
    demo_if = demo_sst['instantaneous_frequency']
    demo_n,demo_channels = demo_if['frequencies'].shape
    demo_frequency_df = pd.DataFrame(dict(sample_id=np.repeat(demo_coords['sample_id'].to_numpy(),demo_channels),
        channel_id=np.tile(np.arange(demo_channels),demo_n),raw_frequency=demo_if['raw_frequency'].ravel(),
        frequency_cycles_per_sample=demo_if['frequencies'].ravel(),period_samples=demo_if['period_samples'].ravel(),
        model_angular_frequency=demo_if['model_angular_frequency'].ravel(),
        coefficient_threshold=np.repeat(demo_if['threshold'],demo_channels),
        frequency_status=demo_if['status'].ravel(),frequency_reason=demo_if['reason'].ravel(),
        assigned_bin_id=demo_sst['assigned_bins'].ravel(),reassignment_status=demo_sst['reassignment_status'].ravel(),
        reassignment_reason=demo_sst['reassignment_reason'].ravel()))
    demo_spectrum_df = pd.DataFrame(np.column_stack([demo_sst['spectrum'].real,demo_sst['spectrum'].imag]),
        columns=[f'sst_{component}_{j:03d}' for component in ['real','imag'] for j in range(128)])
    demo_point_df = pd.concat([demo_coords,demo_sst['diagnostics_df'],demo_spectrum_df],axis=1)
    demo_frequency_frames[demo_kind],demo_point_frames[demo_kind] = demo_frequency_df,demo_point_df
    demo_identity_base = dict(artifact_role='method_demo',method_version=SYNCHROSQUEEZING_VERSION,
        producer_notebook=demo_producer_notebook.relative_to(candidate_root).as_posix(),method_source_tag='demo-dependency',
        method_source_sha256=hashlib.sha256(demo_method_source.encode()).hexdigest(),
        dependency_method_source_sha256=dict(wavelet_transform=demo_upstream_source_hash),
        dependency_source_sha256={role:hashlib.sha256((candidate_root/relative).read_bytes()).hexdigest()
            for role,relative in [('notebook_loader','R04_Research/a00_notebook_loader.py'),('data_contracts','config/data_contracts.py')]},
        upstream_demo_identity=demo_upstream_identities[demo_kind],
        scope=demo_upstream_identities[demo_kind]['coefficients']['scope'],timezone='Asia/Shanghai',
        input_rows=demo_n,parameters=dict(kernel_type=demo_kind,**demo_sst['parameters'],frequency_grid=demo_grid.parameters),
        actual_sample_at_start=demo_upstream_identities[demo_kind]['coefficients']['actual_sample_at_start'],
        actual_sample_at_end=demo_upstream_identities[demo_kind]['coefficients']['actual_sample_at_end'],
        bank_signature=demo_sst['bank'].signature,frequency_grid_signature=demo_grid.signature,
        runtime=dict(python=sys.version.split()[0],numpy=np.__version__,pandas=pd.__version__,pyarrow=pa.__version__,scipy=scipy.__version__,platform=platform.platform()),
        status_counts={str(k):int(v) for k,v in demo_sst['diagnostics_df']['sst_status'].value_counts().items()})
    demo_config_token = f"{demo_kernel_tokens[demo_kind]}_{demo_period_token}_formal_rel0.01_B128_linear_P8-256"
    for demo_role,demo_frame,demo_schema,demo_sort_keys in [
        ('grid',demo_grid.bins_df,SST_GRID_SCHEMA,[('bin_id','ascending')]),
        ('frequencies',demo_frequency_df,SST_FREQUENCY_SCHEMA,[('sample_id','ascending'),('channel_id','ascending')]),
        ('spectrum',demo_point_df,demo_point_schema,[('sample_id','ascending')])]:
        demo_table = pandas_to_arrow(demo_frame,demo_schema).sort_by(demo_sort_keys)
        demo_identity = dict(demo_identity_base,output_role=demo_role,output_rows=demo_table.num_rows,output_sort_keys=demo_sort_keys,
            output_schema_sha256=hashlib.sha256(demo_schema.serialize().to_pybytes()).hexdigest(),
            output_content_sha256=arrow_content_sha256(demo_table,demo_schema,demo_sort_keys))
        demo_identity = json.loads(json.dumps(demo_identity,ensure_ascii=False,allow_nan=False))
        demo_path = demo_output_dir/f'c02_Synchrosqueezing_RB_{demo_range_token}_{demo_input_token}_{demo_config_token}_{demo_role}_demo.parquet'
        demo_staging = demo_path.with_name(demo_path.name+'.staging')
        if demo_staging.exists():
            raise RuntimeError(f'存在未处理 staging：{demo_staging}')
        demo_preserve = False
        if demo_path.exists():
            demo_existing_schema = pq.read_schema(demo_path)
            demo_existing_identity = json.loads(demo_existing_schema.metadata[b'demo_identity'])
            demo_existing_base = demo_existing_schema.with_metadata({k:v for k,v in demo_existing_schema.metadata.items() if k!=b'demo_identity'})
            assert arrow_content_sha256(demo_path,demo_existing_base,demo_existing_identity['output_sort_keys'])==demo_existing_identity['output_content_sha256']
            demo_preserve = demo_existing_base.equals(demo_schema,check_metadata=True) and {k:v for k,v in demo_existing_identity.items() if k!='generated_at_utc'}==demo_identity
        if not demo_preserve:
            demo_installed = dict(demo_identity,generated_at_utc=datetime.now(timezone.utc).isoformat())
            demo_metadata = {**demo_schema.metadata,b'demo_identity':json.dumps(demo_installed,ensure_ascii=False,allow_nan=False,sort_keys=True).encode()}
            pq.write_table(demo_table.replace_schema_metadata(demo_metadata),demo_staging,compression='zstd')
            assert arrow_content_sha256(demo_staging,demo_schema,demo_sort_keys)==demo_identity['output_content_sha256']
            os.replace(demo_staging,demo_path)
        demo_installed_schema = pq.read_schema(demo_path)
        assert demo_installed_schema.with_metadata({k:v for k,v in demo_installed_schema.metadata.items() if k!=b'demo_identity'}).equals(demo_schema,check_metadata=True)
        demo_installed_identity = json.loads(demo_installed_schema.metadata[b'demo_identity'])
        assert {k:v for k,v in demo_installed_identity.items() if k!='generated_at_utc'}==demo_identity
        assert arrow_content_sha256(demo_path,demo_schema,demo_sort_keys)==demo_identity['output_content_sha256']
        print(f'{demo_kind} {demo_role} 已保存并复读验收：{demo_table.num_rows:,} 行',flush=True)


causal_gammatone grid 已保存并复读验收：128 行
causal_gammatone frequencies 已保存并复读验收：127,896 行
causal_gammatone spectrum 已保存并复读验收：21,316 行
morlet grid 已保存并复读验收：128 行
morlet frequencies 已保存并复读验收：127,896 行
morlet spectrum 已保存并复读验收：21,316 行


In [9]:
display(demo_grid.bins_df.head(8).rename(columns=SST_COLUMN_LABELS).rename_axis('row（频率网格行）'))
for demo_kind,demo_sst in demo_results.items():
    print(f'核类型：{demo_kind}')
    display(demo_frequency_frames[demo_kind].tail(12).rename(columns=SST_COLUMN_LABELS).rename_axis('row（瞬时频率与归属行）'))
    demo_preview_columns = ['sample_id','sample_at','sample_side','coefficient_threshold','ready_scale_count','reassigned_scale_count','sst_status','sst_reason']
    display(demo_point_frames[demo_kind][demo_preview_columns].tail(8).rename(columns=SST_COLUMN_LABELS).rename_axis('row（末端SST状态行）'))
    display(demo_sst['diagnostics_df'].groupby(['sst_status','sst_reason'],dropna=False).size().rename('采样点数').reset_index()
        .rename(columns=SST_COLUMN_LABELS).rename_axis('row（完整真实状态汇总行）'))
print('c02 真实 demo 已完成；独立绘图与序列调度归后续 c03。')


核类型：causal_gammatone
核类型：morlet
c02 真实 demo 已完成；独立绘图与序列调度归后续 c03。


,bin_id（频率区间编号）,frequency_left（区间左频率）,frequency_right（区间右频率）,frequency_center（区间中心周频率）,frequency_width（区间频率宽度）,period_samples（周期采样点数）
row（频率网格行）,,,,,,
0,0,0.003906,0.004852,0.004379,0.000946,228.348432
1,1,0.004852,0.005798,0.005325,0.000946,187.782235
2,2,0.005798,0.006744,0.006271,0.000946,159.454988
3,3,0.006744,0.007690,0.007217,0.000946,138.553911
4,4,0.007690,0.008636,0.008163,0.000946,122.497196
5,5,0.008636,0.009583,0.009109,0.000946,109.775544
6,6,0.009583,0.010529,0.010056,0.000946,99.447648
7,7,0.010529,0.011475,0.011002,0.000946,90.895978


,sample_id（采样点编号）,channel_id（尺度通道编号）,raw_frequency（未裁剪的周频率估计）,frequency_cycles_per_sample（有效瞬时周频率）,period_samples（周期采样点数）,model_angular_frequency（导数模型角频率）,coefficient_threshold（当前点幅值阈值）,frequency_status（瞬时频率状态）,frequency_reason（瞬时频率原因）,assigned_bin_id（重分配区间编号）,reassignment_status（重分配状态）,reassignment_reason（重分配原因）
row（瞬时频率与归属行）,,,,,,,,,,,,
127884,21315,0,0.129864,0.129864,7.700389,0.864460,0.000044,ok,,-1,excluded,outside_frequency_grid
127885,21315,1,0.060568,0.060568,16.510283,0.385222,0.000044,ok,,59,ok,
127886,21315,2,0.030287,0.030287,33.017753,0.190874,0.000044,ok,,27,ok,
127887,21315,3,0.016302,0.016302,61.341899,0.102519,0.000044,ok,,13,ok,
127888,21315,4,0.007434,0.007434,134.511102,0.046720,0.000044,ok,,3,ok,
127889,21315,5,0.004133,0.004133,241.968404,0.025968,0.000044,ok,,0,ok,
127890,21316,0,0.130717,0.130717,7.650121,0.870830,0.000044,ok,,-1,excluded,outside_frequency_grid
127891,21316,1,0.060680,0.060680,16.479915,0.385950,0.000044,ok,,60,ok,
127892,21316,2,0.030334,0.030334,32.966070,0.191175,0.000044,ok,,27,ok,


,sample_id（采样点编号）,sample_at（采样位置时刻）,sample_side（训练测试侧）,coefficient_threshold（当前点幅值阈值）,ready_scale_count（上游就绪尺度数）,reassigned_scale_count（纳入重分配尺度数）,sst_status（同步压缩状态）,sst_reason（同步压缩原因）
row（末端SST状态行）,,,,,,,,
21308,21309,2026-09-29 13:45:00+08:00,test,0.000043,6,6,ok,
21309,21310,2026-09-29 14:45:00+08:00,test,0.000043,6,6,ok,
21310,21311,2026-09-29 21:45:00+08:00,test,0.000043,6,6,ok,
21311,21312,2026-09-29 22:45:00+08:00,test,0.000043,6,6,ok,
21312,21313,2026-09-30 09:45:00+08:00,test,0.000044,6,5,ok,
21313,21314,2026-09-30 11:00:00+08:00,test,0.000044,6,5,ok,
21314,21315,2026-09-30 14:00:00+08:00,test,0.000044,6,5,ok,
21315,21316,2026-09-30 15:00:00+08:00,test,0.000044,6,5,ok,


,sst_status（同步压缩状态）,sst_reason（同步压缩原因）,采样点数
row（完整真实状态汇总行）,,,
0,failed,upstream_failed,1
1,ok,,16055
2,partial,unavailable_or_failed_scales,5074
3,warmup,insufficient_history,186


,sample_id（采样点编号）,channel_id（尺度通道编号）,raw_frequency（未裁剪的周频率估计）,frequency_cycles_per_sample（有效瞬时周频率）,period_samples（周期采样点数）,model_angular_frequency（导数模型角频率）,coefficient_threshold（当前点幅值阈值）,frequency_status（瞬时频率状态）,frequency_reason（瞬时频率原因）,assigned_bin_id（重分配区间编号）,reassignment_status（重分配状态）,reassignment_reason（重分配原因）
row（瞬时频率与归属行）,,,,,,,,,,,,
127884,21315,0,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127885,21315,1,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127886,21315,2,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127887,21315,3,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127888,21315,4,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127889,21315,5,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127890,21316,0,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127891,21316,1,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support
127892,21316,2,NaN,NaN,NaN,NaN,NaN,boundary_incomplete,incomplete_kernel_support,-1,boundary_incomplete,incomplete_kernel_support


,sample_id（采样点编号）,sample_at（采样位置时刻）,sample_side（训练测试侧）,coefficient_threshold（当前点幅值阈值）,ready_scale_count（上游就绪尺度数）,reassigned_scale_count（纳入重分配尺度数）,sst_status（同步压缩状态）,sst_reason（同步压缩原因）
row（末端SST状态行）,,,,,,,,
21308,21309,2026-09-29 13:45:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21309,21310,2026-09-29 14:45:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21310,21311,2026-09-29 21:45:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21311,21312,2026-09-29 22:45:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21312,21313,2026-09-30 09:45:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21313,21314,2026-09-30 11:00:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21314,21315,2026-09-30 14:00:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support
21315,21316,2026-09-30 15:00:00+08:00,test,NaN,0,0,boundary_incomplete,incomplete_kernel_support


,sst_status（同步压缩状态）,sst_reason（同步压缩原因）,采样点数
row（完整真实状态汇总行）,,,
0,boundary_incomplete,incomplete_kernel_support,124
1,failed,upstream_failed,1
2,ok,,17403
3,partial,unavailable_or_failed_scales,3788


## 实际验收状态

2026-10-07 已在 `latitude_env_v2` 逐代码单元格执行上述完整真实 demo，表格与状态输出保存在本 Notebook。两种核各处理 21,316 点：因果核为 16,055 个 ok、5,074 个 partial、186 个 warmup、1 个 failed；Morlet 为 17,403 个 ok、3,788 个 partial、124 个 boundary_incomplete、1 个 failed。两个 failed 均对应上游首个收益率缺失点，没有新增数值失败；partial 表示部分尺度预热或支持不足，不表示已经得到完整尺度谱。

两种核各保存频率网格 128 行、逐尺度频率与归属 127,896 行、复谱与点状态 21,316 行，六份 Parquet 均已复读核对局部 Schema、生成身份与内容摘要。完整真实验收通过复积分恒等式、当前点阈值、范围排除和训练前缀历史不变性；绝对阈值、对数频率网格与 legacy_sum 使用相同真实输入在内存验收。18 项独立合成数学与边界检查通过，覆盖核导数逆映射、实正弦频率、阈值、区间边界、复数抵消、部分尺度、空谱与零谱、数值溢出和参数拒绝。

临时验证与构建／执行工具暂存 `R00_draft_collection_02/tests/test_synchrosqueezing.py`、`R00_draft_collection_02/scripts/*synchrosqueezing*`，不作为算法导入入口。完整序列调度和独立单图绘制归后续 c03，尚未实现。